In [3]:
import os
import json
import pandas as pd
from datetime import datetime

from google.colab import drive
drive.mount('/content/drive')

ROOT = "/content/drive/MyDrive/ENARES_2024_PROJECT"

RAW_DIR = os.path.join(ROOT, "01BasesDatosPrimarias")
LOG_DIR = os.path.join(ROOT, "05Resultados/logs")
REPORT_DIR = os.path.join(ROOT, "04CuestionariosInformes/reportes")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
# manifest (latest)
manifest_files = [f for f in os.listdir(RAW_DIR) if "manifest" in f]
manifest_path = os.path.join(RAW_DIR, sorted(manifest_files)[-1])

with open(manifest_path, "r") as f:
    manifest = json.load(f)

# logs
log_files = [f for f in os.listdir(RAW_DIR) if "log_ingesta" in f]
log_path = os.path.join(RAW_DIR, sorted(log_files)[-1])

with open(log_path, "r") as f:
    log_lines = f.readlines()

# catalogue
catalogue_files = [f for f in os.listdir(LOG_DIR) if "catalogo" in f]
catalogue_path = os.path.join(LOG_DIR, sorted(catalogue_files)[-1])
catalogue = pd.read_csv(catalogue_path)

# CRS04 outputs
vars_df = pd.read_csv(os.path.join(LOG_DIR, "ENARES_2024_CRS04_variables_stage1.csv"))
values_df = pd.read_csv(os.path.join(LOG_DIR, "ENARES_2024_CRS04_value_labels_stage1.csv"))
validation_df = pd.read_csv(os.path.join(LOG_DIR, "ENARES_2024_CRS04_validacion_stage1.csv"))

missing_path = os.path.join(LOG_DIR, "ENARES_2024_CRS04_missing_codes_stage1.csv")

if os.path.exists(missing_path) and os.path.getsize(missing_path) > 0:
    missing_df = pd.read_csv(missing_path)
else:
    missing_df = pd.DataFrame()  # empty safe dataframe

In [5]:
checks = {
    "manifest_exists": manifest is not None,
    "log_exists": len(log_lines) > 0,
    "catalogue_rows": len(catalogue),
    "variables_rows": len(vars_df),
    "values_rows": len(values_df),
    "validation_rows": len(validation_df),
}

checks

{'manifest_exists': True,
 'log_exists': True,
 'catalogue_rows': 66,
 'variables_rows': 1299,
 'values_rows': 3317,
 'validation_rows': 4}

In [6]:
modules_processed = len(manifest)
modules_failed = len([m for m in manifest if m["status"] != "success"])

modules_processed, modules_failed

(22, 0)

In [7]:
crs04_modules = validation_df["module_id"].unique().tolist()
crs04_rows = validation_df["n_rows"].sum()
crs04_cols_sample = validation_df["n_columns"].iloc[0]

crs04_modules, crs04_rows, crs04_cols_sample

(['976-Modulo1959', '976-Modulo1960', '976-Modulo1961', '976-Modulo1962'],
 np.int64(75228),
 np.int64(147))

In [8]:
report_path = os.path.join(REPORT_DIR, "ENARES_2024_STAGE1_ingestion_report.md")

with open(report_path, "w") as f:

    f.write("# ENARES 2024 - Stage 1 Ingestion Report\n\n")

    f.write("## Account Used\n")
    f.write("anacordero.001@gmail.com\n\n")

    f.write("## Objective\n")
    f.write("Reproducible ingestion of ENARES 2024 CRS04 dataset from INEI SPSS ZIP source.\n\n")

    f.write("## Official Data Source\n")
    f.write("https://proyectos.inei.gob.pe/microdatos/\n\n")

    f.write("## Official Source Package and Raw Format\n")
    f.write(
        "- Data downloaded as SPSS ZIP packages from INEI\n"
        "- ZIP files preserved intact in Google Drive\n"
        "- .sav files are the raw analytical source\n"
        "- CSV/Stata formats were NOT used as primary source\n\n"
    )

    f.write("## Modules Processed\n")
    f.write(f"{modules_processed}\n\n")

    f.write("## Modules Failed\n")
    f.write(f"{modules_failed}\n\n")

    f.write("## Integrity Checks\n")
    for k, v in checks.items():
        f.write(f"- {k}: {v}\n")
    f.write("\n")

    f.write("## CRS04 Identification\n")
    for m in crs04_modules:
        f.write(f"- {m}\n")
    f.write("\n")

    f.write("## CRS04 Initial Validation\n")
    f.write(f"- Total rows (approx): {crs04_rows}\n")
    f.write(f"- Columns per module: {crs04_cols_sample}\n\n")

    f.write("## Runtime Environment\n")
    f.write("- Google Colab\n")
    f.write("- Python + pandas + pyreadstat\n")
    f.write("- Google Drive API v3\n\n")

    f.write("## Issues Found\n")
    f.write("- None critical during ingestion stage\n\n")

    f.write("## Pending Questions for Supervisor\n")
    f.write("- Confirmation of CRS04 module boundaries\n")
    f.write("- Validation of derived disability variable definition (C4P130_1–C4P130_6)\n\n")

    f.write("## Stage 1 Decision and Next Step\n")
    f.write("Proceed to Stage 2 (Cloud Storage / BigQuery ingestion preparation)\n")

print("Report saved:", report_path)

Report saved: /content/drive/MyDrive/ENARES_2024_PROJECT/04CuestionariosInformes/reportes/ENARES_2024_STAGE1_ingestion_report.md
